# EXP-06C — Verificação de retiradas históricas (Privacy Pools v1)

**Objetivo:** localizar até duas transações `Withdrawn` em cada uma de duas janelas de blocos pré-fixadas e conferir a transação e os logs pelo Blockscout. Não repetir EXP-06A/B; não treinar nem avaliar RF; **não atribuir rótulos de mixer**.

**Janelas diagnósticas fixas:** blocos `22.153.714–22.253.714` (após lançamento de 2025) e `24.233.073–24.240.273` (vizinha do dia de teste explorado em 2026). A segunda janela **não é holdout cego**, mesmo que obtenha transações novas. A seleção dos limites não representa amostra aleatória de Ethereum. As buscas podem parar após encontrar dois candidatos por janela ou diante de erros/limites de API.

**Ressalva:** `Withdrawn` difere de `Ragequit`. O identificador `topic0` é uma assinatura de evento observada previamente; a confirmação no receipt não demonstra por si só anonimato ou desvinculação. Os nomes decodificados devem ser comparados com a implementação do período antes da classificação.

**Executar:** Google Colab → Ambiente de execução → Executar tudo. Será criada uma nova pasta de resultados no Drive; os arquivos históricos permanecem intocados.

In [ ]:
%pip -q install requests google-api-python-client google-auth-httplib2 google-auth-oauthlib
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import requests, subprocess, sys, hashlib, io, json
api=build('drive','v3')
base=Path('/content/EXP-06C');base.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
print('Drive autenticado; EXP-06C preparado.')

## 1. Recuperar EXP-06B e validar SHA-256
A auditoria só prossegue se os dados EXP-06B forem exatamente os que aprovamos.

In [ ]:
original=base/'evidencias_exp06b.json'
req=api.files().get_media(fileId='1tqOhpMwf_qce64d7-7F7YuO1H20naeAF')
with original.open('wb') as target:
    dl=MediaIoBaseDownload(target,req);done=False
    while not done:_,done=dl.next_chunk()
expected='61e60f28fd23af38e09f8ae2a6ae23d56699536e2667f7d0d1f875adebbc207f'
assert sha(original)==expected,'EXP06B divergente — interromper'
print('EXP06B íntegro:',sha(original))

## 2. Baixar o script do commit fixado e conferir o hash
Se houver divergência, a execução deve parar antes de consultar a blockchain.

In [ ]:
commit='9a922526196bdc0face23858d1923f7b834c9a28'
url=f'https://raw.githubusercontent.com/BorgesGusta/ethereum-mixer-detection/{commit}/scripts/exp06c_retirada_historica.py'
r=requests.get(url,timeout=40);r.raise_for_status()
script=base/'exp06c_retirada_historica.py';script.write_bytes(r.content)
expected_script='fdd82442ca69b8af77cfd8840d7f70d56571e8a0e336b6b20539d4a6a7bebd6a'
assert sha(script)==expected_script,'Código difere do script congelado — interromper'
print('Script conferido:',sha(script))

## 3. Executar somente o piloto EXP-06C

A consulta usa o endpoint histórico de logs com filtro de **endereço + topic0 + intervalo de blocos**, não a listagem de logs recentes. Em seguida confere até duas transações candidatas por janela e seus logs de receipt. O total de chamadas é limitado e os erros de API são preservados.

In [ ]:
out=base/'saida'
subprocess.run([sys.executable,str(script),'--exp06b',str(original),'--saida',str(out)],check=True)
d=json.loads((out/'evidencias_exp06c.json').read_text(encoding='utf-8'))
for w in d['janelas']:
    print(w['janela'],w['status_cobertura'],'candidatos:',w['n_candidatos_encontrados_ate_parada'])
    for t in w['transacoes_verificadas']:
        print(' ',t['tx_hash'], 'log de retirada conferido:',t['confirmed_withdraw_log'])
print('NENHUM rótulo de mixer gerado.')

## 4. Criar pasta nova em `results` e salvar manifesto + evidências

O script não sobrescreve outputs anteriores. Confirme que a pasta e seus dois arquivos aparecem no Drive.

In [ ]:
parent='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'
name='EXP-06C_retiradas_historicas_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
folder=api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent]},fields='id,name').execute()
for p in sorted(out.iterdir()):
    saved=api.files().create(body={'name':p.name,'parents':[folder['id']]},media_body=MediaFileUpload(str(p),mimetype='application/json',resumable=True),fields='id,name').execute()
    print('Salvo:',saved['name'],saved['id'])
print('RESULTADOS: https://drive.google.com/drive/folders/'+folder['id'])

## Depois da execução

Responda no ChatGPT: **“EXP-06C executado, confira o Drive”**. A auditoria posterior verificará a integridade, a cobertura real, os recibos e se há evidência histórica de retirada. **Nenhum resultado automático demonstra, sozinho, generalização ou melhoria do RF.**